# Depth Pro (Apple)

Metric depth. **Run this first** — it also selects the 200 samples used by all other notebooks.

**Runtime:** GPU (L4 or A100)

## Setup + sample selection

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, random, time, gc, sys, json, importlib, subprocess
import numpy as np
import cv2
import torch
from PIL import Image

DATASET_IMAGES = '/content/drive/MyDrive/Corn Seed Dataset/test/images'
DATASET_LABELS = '/content/drive/MyDrive/Corn Seed Dataset/test/labels'
SAVE_DIR       = '/content/drive/MyDrive/Corn Seed Dataset/depth_comparison_outputs'
SAMPLE_FILE    = os.path.join(SAVE_DIR, 'sample_images.txt')

assert os.path.isdir(DATASET_IMAGES), f'Dataset not found: {DATASET_IMAGES}'
os.makedirs(SAVE_DIR, exist_ok=True)

def save_depth(depth_np, stem, model_name):
    d = np.array(depth_np, dtype=np.float32)
    while d.ndim > 2: d = d[0]
    d_norm = (d - d.min()) / (d.max() - d.min() + 1e-8)
    for sub, img in [('depth', (d_norm*65535).astype(np.uint16)),
                     ('vis',   cv2.applyColorMap((d_norm*255).astype(np.uint8), cv2.COLORMAP_INFERNO))]:
        p = os.path.join(SAVE_DIR, model_name, sub)
        os.makedirs(p, exist_ok=True)
        cv2.imwrite(os.path.join(p, f'{stem}.png'), img)

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('Setup done.')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Select 200 random samples and save to Drive for reuse across all notebooks
all_imgs = sorted(f for f in os.listdir(DATASET_IMAGES) if f.endswith('.jpg'))
random.seed(42)
SAMPLES = sorted(random.sample(all_imgs, 200), key=lambda x: int(x.split('.')[0]))
with open(SAMPLE_FILE, 'w') as f:
    f.write('\n'.join(SAMPLES))
print(f'Selected {len(SAMPLES)} samples -> {SAMPLE_FILE}')

## Install + run

In [ ]:
!pip install -q 'transformers>=4.48'
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

MODEL_NAME = 'depth_pro'
processor = AutoImageProcessor.from_pretrained('apple/DepthPro-hf', trust_remote_code=True)
model = AutoModelForDepthEstimation.from_pretrained(
    'apple/DepthPro-hf', trust_remote_code=True, dtype=torch.float16).cuda().eval()

times = []
for img_name in SAMPLES:
    stem = img_name.split('.')[0]
    image = Image.open(os.path.join(DATASET_IMAGES, img_name)).convert('RGB')
    inputs = processor(images=image, return_tensors='pt').to('cuda')
    t0 = time.time()
    with torch.no_grad(): outputs = model(**inputs)
    torch.cuda.synchronize(); times.append(time.time()-t0)
    post = processor.post_process_depth_estimation(outputs, target_sizes=[(image.height, image.width)])
    save_depth(post[0]['predicted_depth'].cpu().float().numpy(), stem, MODEL_NAME)

print(f'Done -- {len(times)} images, avg {np.mean(times):.3f}s/img')
del model, processor; clear_gpu()

## Confirm saved

In [ ]:
model_dir = os.path.join(SAVE_DIR, MODEL_NAME)
n_depth = len(os.listdir(os.path.join(model_dir, 'depth')))
n_vis   = len(os.listdir(os.path.join(model_dir, 'vis')))
print(f'{MODEL_NAME}: {n_depth} depth maps, {n_vis} visualizations saved to Drive')